In [ ]:
import torch
import torch.nn as nn
from torchvision import models

In [ ]:
# replace the classification head
model = models.resnet50(pretrained=True)

# 1. Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# 2. Replace classification head
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)

# 3. Optimizer for trainable params only
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=1e-3
)

In [ ]:
# progressive unfreezing
# Unfreeze last 2 layers
for layer in model.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Create parameter groups
optimizer = torch.optim.AdamW([
    {"params": model.encoder.layer[-2:].parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-3}
])


In [ ]:
# unfreeze top k layers and return total number of trainable parameters
def freeze_except_top_k(model, k):
    layers = list(model.children())

    # Freeze all
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze top-k layers
    for layer in layers[-k:]:
        for param in layer.parameters():
            param.requires_grad = True

    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    return trainable

In [ ]:
# batch norm specific
# keep it in train() mode, so the parameter will be update, and running stats will update
for module in model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.train()  # keep running stats updating
    else:
        for param in module.parameters():
            param.requires_grad = False

BatchNorm has two different kinds of state:  
**Trainable parameters**
- weight (γ)
- bias (β)  
→ controlled by param.requires_grad

**Running statistics**
- running_mean 
- running_var  
→ controlled by module.train() vs module.eval()

In [ ]:
# batch norm, to complete freeze it
# put it in eval() mode and set requires_grad to False for parameters
for m in model.modules():
    if isinstance(m, nn.BatchNorm2d):
        m.eval()
        for p in m.parameters():
            p.requires_grad = False